# Test notebook

In [ ]:
import time
from pathlib import Path

import cv2
from tqdm.auto import tqdm
from ultralytics import YOLO

ROOT = Path.cwd()
assert (ROOT / "split_dataset.py").exists(), f"run the notebook from the task folder, not {ROOT}"

TEST_DIR = ROOT / "input" / "test"
OUT_DIR = ROOT / "output" / "test_preds"
RUNS_DIR = ROOT / "output" / "runs"
WEIGHTS = ROOT / "weights" / "yolo26_shufflenet_640_best.pt" # Do not forget to change, for newer model
CONF = 0.429
DATA_YAML = ROOT / "input" / "train" / "data.yaml"
IMGSZ = 640  # must match training imgsz

model = YOLO(WEIGHTS)

# Confidence threshold (optional)
`CONF` in the cell above is the threshold with the best F1 on the **val** split (the test clips were never used to pick it).
This cell shows how it was chosen and checks that it still matches `WEIGHTS`. It needs the val split in `input/train/`
(built by the *Dataset preparation* cell of `train-notebook.ipynb`) and is skipped without it.

In [ ]:
if DATA_YAML.exists():
    metrics = model.val(
        data=str(DATA_YAML), split="val", imgsz=IMGSZ, plots=False,
        project=str(RUNS_DIR), name="conf_pick", exist_ok=True,
    )
    f1 = metrics.box.f1_curve.mean(0)
    val_conf = float(metrics.box.px[f1.argmax()])
    print(f"best val F1 = {f1.max():.3f} at conf = {val_conf:.3f}")
    # px is a 1000-point grid over [0, 1]: a gap of more than one step means CONF was picked for other weights
    assert abs(val_conf - CONF) <= 1e-3, f"CONF = {CONF} is stale for {WEIGHTS.name}: set it to {val_conf:.3f}"
else:
    print(f"skipped: no {DATA_YAML.relative_to(ROOT)}, using CONF = {CONF}")

# Inference on test clips

In [ ]:
BOX_COLOR = (0, 255, 0)  # BGR
FONT = cv2.FONT_HERSHEY_SIMPLEX


def draw_label(frame, text, x, y, scale=0.5, bg=BOX_COLOR):
    """Black text on a filled background, bottom-left corner at (x, y)."""
    (tw, th), baseline = cv2.getTextSize(text, FONT, scale, 1)
    y = max(y, th + baseline)  # boxes touching the top edge: keep their label inside the frame
    cv2.rectangle(frame, (x, y - th - baseline), (x + tw, y), bg, -1)
    cv2.putText(frame, text, (x, y - baseline), FONT, scale, (0, 0, 0), 1, cv2.LINE_AA)


def annotate_clip(src: Path, dst: Path, conf: float, imgsz: int) -> dict:
    """Run the detector on every frame of src and write the clip with boxes drawn on top to dst."""
    cap = cv2.VideoCapture(str(src))
    assert cap.isOpened(), f"cannot open {src}"
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # container metadata: only used for the progress bar
    out = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))  # source fps keeps 59.94 fps timing
    assert out.isOpened(), f"cannot write {dst}"

    frames, boxes_total = 0, 0
    t0 = time.perf_counter()
    try:
        with tqdm(total=n_frames, desc=src.stem) as pbar:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                r = model.predict(frame, conf=conf, imgsz=imgsz, verbose=False)[0]  # BGR ndarray, as cv2 gives it
                for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.int().tolist(), r.boxes.conf.tolist()):
                    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)  # 1 px is hard to see at 1080p
                    draw_label(frame, f"{c:.2f}", x1, y1)
                draw_label(frame, f"vehicles: {len(r.boxes)}  frame: {frames}", 10, 40, scale=1.0, bg=(255, 255, 255))
                out.write(frame)
                frames += 1
                boxes_total += len(r.boxes)
                pbar.update()
    finally:
        cap.release()
        out.release()

    seconds = time.perf_counter() - t0
    return {
        "frames": frames,
        "seconds": seconds,
        "fps": frames / seconds,
        "boxes_per_frame": boxes_total / max(frames, 1),
    }

OUT_DIR.mkdir(parents=True, exist_ok=True)
clips = sorted(TEST_DIR.glob("*.mp4"))
assert clips, f"no .mp4 in {TEST_DIR}"

for src in clips:
    dst = OUT_DIR / f"{src.stem}_pred.mp4"
    s = annotate_clip(src, dst, CONF, IMGSZ)
    # pipeline fps = decode + predict + draw + encode, not model-only speed
    print(f"{src.name}: {s['frames']} frames in {s['seconds']:.1f}s ({s['fps']:.0f} pipeline fps), "
          f"{s['boxes_per_frame']:.1f} boxes/frame -> {dst}")

# Metrics

In [3]:
import math

import numpy as np
import pandas as pd
from ultralytics.utils.metrics import compute_ap

# ---------------------------------------------------------------- assumptions ---
# Range comes from box size with a pinhole camera:  distance = f_px * CAR_LEN_M / longer box side in px.
#  1. Every vehicle is CAR_LEN_M long, and the longer side of its box spans that length (top-down view:
#     holds at any heading, a car rotated 45 deg still has a box side close to its length).
#  2. f_px comes from the FOV of the UNCROPPED frame. The eval clips are 1080x1080 crops of (assumed)
#     1920x1080 video: cropping drops pixels but keeps f_px, so the FOV must be applied to the full frame.
#  3. Range barely changes within a clip (8-30 s), so each vehicle gets ONE band, from its median box size.
#     Per-box bands would move a car to the far band whenever it is half out of frame or occluded.
#  4. An unmatched prediction that overlaps a vehicle (half-car box, duplicate) is a localization error:
#     it costs precision in that vehicle's band, but it is not a false alarm. A false alarm is a box on
#     empty background, binned at the range the system would report for it (its own box size), and it
#     counts once per event (overlapping boxes merged, then linked over frames), not once per frame.
#  5. Time to first detection (TTFD) runs from a vehicle's first labelled frame (clip start for vehicles
#     already in view) to its first matched prediction at CONF.
CAR_LEN_M = 4.5
BANDS = [("0-100 m", 0.0, 100.0), ("100-400 m", 100.0, 400.0)]
IOU_THR = 0.5   # prediction <-> GT match
LINK_IOU = 0.3  # box <-> same object in a later frame
LINK_GAP = 15   # frames an object may vanish (blur, occlusion) and keep its identity: 0.5 s at 30 fps
CLIPS = {
    # folder in input/test/annotated -> source video (for fps), full uncropped frame size and its FOV:
    # hfov_deg or dfov_deg (diagonal, what drone spec sheets usually quote). If the clips were shot at
    # different zoom levels, they need different FOV values here.
    "close-up":  dict(video="eval_close-up_cropped.mp4",  full_wh=(1920, 1080), dfov_deg=84.0),
    "wide-shot": dict(video="eval_wide-shot_cropped.mp4", full_wh=(1920, 1080), dfov_deg=84.0),
}


# --------------------------------------------------------------- geometry ---
def focal_px(width, height, hfov_deg=None, dfov_deg=None):
    """Focal length in pixels from the FOV of a width x height frame (pinhole model)."""
    if hfov_deg is not None:
        tan_half_h = math.tan(math.radians(hfov_deg) / 2)
    else:  # diagonal -> horizontal: tangent scales with the width's share of the diagonal
        tan_half_h = math.tan(math.radians(dfov_deg) / 2) * width / math.hypot(width, height)
    return (width / 2) / tan_half_h


def distance_m(boxes, f_px):
    """d = f * L / s for every x1 y1 x2 y2 box, with the longer box side standing in for car length."""
    side = np.maximum(boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1])
    return f_px * CAR_LEN_M / np.maximum(side, 1e-6)


def band_of(dist):
    """Index of the band this distance falls into, or -1 (beyond 400 m: not scored)."""
    for i, (_, lo, hi) in enumerate(BANDS):
        if lo <= dist < hi:
            return i
    return -1


# ------------------------------------------------------------------- I/O ---
def read_gt(txt_path, W, H):
    """YOLO txt -> (N, 4) array of x1 y1 x2 y2 in pixels. Missing/empty file = no boxes."""
    lines = txt_path.read_text().splitlines() if txt_path.exists() else []
    rows = [list(map(float, ln.split())) for ln in lines if ln.strip()]
    if not rows:
        return np.zeros((0, 4))
    a = np.array(rows)
    cx, cy, w, h = a[:, 1] * W, a[:, 2] * H, a[:, 3] * W, a[:, 4] * H
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)


def predict_frames(images):
    """(N, 5) array of x1 y1 x2 y2 conf per image, in the order of `images`."""
    by_stem = {}
    # conf=0.001: keep low-score boxes, AP needs the whole precision-recall curve
    for r in model.predict(source=str(images[0].parent), imgsz=IMGSZ, conf=0.001, stream=True, verbose=False):
        by_stem[Path(r.path).stem] = np.column_stack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()])
    return [by_stem[p.stem] for p in images]


# --------------------------------------------------------------- matching ---
def iou_matrix(a, b):
    """IoU between every box in a (N,4) and every box in b (M,4) -> (N,M)."""
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_frame(gt, pred):
    """Greedy matching, most confident prediction first.
    Returns, for every prediction, the index of its matched GT box or -1, plus the (pred, GT) IoU matrix."""
    pred_to_gt = np.full(len(pred), -1)
    ious = iou_matrix(pred[:, :4], gt)
    gt_taken = np.zeros(len(gt), dtype=bool)
    for p in np.argsort(-pred[:, 4]):
        cand = np.where(~gt_taken & (ious[p] >= IOU_THR))[0]
        if len(cand):
            g = cand[np.argmax(ious[p, cand])]
            gt_taken[g] = True
            pred_to_gt[p] = g
    return pred_to_gt, ious


def suppress_overlaps(pred):
    """Keep the most confident box of every group overlapping by IoU >= LINK_IOU: one spot, one alarm."""
    pred = pred[np.argsort(-pred[:, 4])]
    ious = iou_matrix(pred[:, :4], pred[:, :4])
    keep = []
    for i in range(len(pred)):
        if not any(ious[i, k] >= LINK_IOU for k in keep):
            keep.append(i)
    return pred[keep]


def link_tracks(frames):
    """Give boxes an identity across frames: a box continues the track whose latest box it overlaps most
    (IoU >= LINK_IOU, greedy), if that track was seen within the last LINK_GAP frames; else it starts a
    new track. frames: list of (N, 4) arrays -> list of (N,) track-id arrays."""
    last = []  # track id -> (frame, box) of its latest box
    ids_per_frame = []
    for t, boxes in enumerate(frames):
        ids = np.full(len(boxes), -1)
        alive = [k for k, (seen, _) in enumerate(last) if t - seen - 1 <= LINK_GAP]
        if alive and len(boxes):
            ious = iou_matrix(boxes, np.array([last[k][1] for k in alive]))
            for i, j in sorted(zip(*np.nonzero(ious >= LINK_IOU)), key=lambda ij: -ious[ij]):
                if ids[i] < 0 and alive[j] not in ids:
                    ids[i] = alive[j]
        for i in np.where(ids < 0)[0]:
            ids[i] = len(last)
            last.append(None)
        for i, k in enumerate(ids):
            last[k] = (t, boxes[i])
        ids_per_frame.append(ids)
    return ids_per_frame


def track_bands(frames, f_px):
    """Link boxes into tracks and give each track one band, from its median distance.
    Returns the track ids per frame and {track id: band index or -1}."""
    ids_per_frame = link_tracks(frames)
    ids = np.concatenate(ids_per_frame)
    dist = np.concatenate([distance_m(b, f_px) for b in frames])
    return ids_per_frame, {k: band_of(np.median(dist[ids == k])) for k in np.unique(ids)}


# ---------------------------------------------------------------- scoring ---
def score_clip(name, cfg):
    """Raw counts per band: GT boxes, (conf, is_tp) per prediction, TTFD per vehicle, false-alarm events."""
    folder = TEST_DIR / "annotated" / name
    images = sorted((folder / "images" / "train").glob("*.png"))
    cap = cv2.VideoCapture(str(TEST_DIR / cfg["video"]))
    fps, n_video = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    # the frame index is the clock for TTFD and false alarms/min, so every video frame must be labelled
    assert len(images) == n_video, f"{name}: {len(images)} labelled frames, {n_video} video frames"
    H, W = cv2.imread(str(images[0])).shape[:2]
    f = focal_px(*cfg["full_wh"], cfg.get("hfov_deg"), cfg.get("dfov_deg"))

    gts = [read_gt(folder / "labels" / "train" / f"{p.stem}.txt", W, H) for p in images]
    preds = predict_frames(images)
    gt_ids, vehicle_band = track_bands(gts, f)

    bands = [dict(n_gt=0, scored=[], ttfd=[], fa=0, minutes=n_video / fps / 60) for _ in BANDS]
    first_seen, first_hit, fa_boxes = {}, {}, []
    n_dup = n_bad = 0  # unmatched predictions at CONF on a vehicle: IoU >= IOU_THR (duplicate) or below
    for t, (gt, pred, ids) in enumerate(zip(gts, preds, gt_ids)):
        for k in ids:
            first_seen.setdefault(k, t)
            if vehicle_band[k] >= 0:
                bands[vehicle_band[k]]["n_gt"] += 1
        pred_to_gt, ious = match_frame(gt, pred)
        fa = []
        for p, g in enumerate(pred_to_gt):
            conf = pred[p, 4]
            if g >= 0:  # TP: band of the vehicle it found
                b, is_tp = vehicle_band[ids[g]], True
                if conf >= CONF:
                    first_hit.setdefault(ids[g], t)
            elif len(gt) and ious[p].max() > 0:  # duplicate / badly placed box: band of that vehicle
                b, is_tp = vehicle_band[ids[ious[p].argmax()]], False
                if conf >= CONF:
                    n_dup += ious[p].max() >= IOU_THR
                    n_bad += ious[p].max() < IOU_THR
            else:  # box on empty background: band of its own size, the range the system would report
                b, is_tp = band_of(distance_m(pred[p:p + 1], f)[0]), False
                if conf >= CONF:
                    fa.append(pred[p])
            if b >= 0:
                bands[b]["scored"].append((conf, is_tp))
        fa_boxes.append(suppress_overlaps(np.array(fa).reshape(-1, 5))[:, :4])

    for k, t0 in first_seen.items():
        if vehicle_band[k] >= 0:
            bands[vehicle_band[k]]["ttfd"].append((first_hit[k] - t0) / fps if k in first_hit else None)
    _, fa_band = track_bands(fa_boxes, f)
    for b in fa_band.values():
        if b >= 0:
            bands[b]["fa"] += 1

    # sanity check: if a band has (almost) no vehicles, its numbers are noise
    edges = sorted({e for _, lo, hi in BANDS for e in (lo, hi) if e > 0})
    edges = ", ".join(f"{f * CAR_LEN_M / d:.1f} px @ {d:.0f} m" for d in edges)
    per_band = np.bincount([b + 1 for b in vehicle_band.values()], minlength=len(BANDS) + 1)  # [beyond, band0, ...]
    print(f"[{name}] {W}x{H} crop of {cfg['full_wh'][0]}x{cfg['full_wh'][1]}, {fps:.2f} fps, {n_video / fps:.1f} s, "
          f"f = {f:.0f} px, longer-side thresholds: {edges}")
    print("    vehicles per band: " + ", ".join(f"{n} {band}" for (band, _, _), n in zip(BANDS, per_band[1:]))
          + f", {per_band[0]} beyond {BANDS[-1][2]:.0f} m")
    print(f"    unmatched predictions @ CONF: {n_dup} duplicates of a found vehicle, {n_bad} badly placed on a vehicle, "
          f"{sum(map(len, fa_boxes))} on background -> {len(fa_band)} false-alarm events")
    return bands


def merge(bands):
    """Pool the raw counts of one band over several clips."""
    return dict(n_gt=sum(b["n_gt"] for b in bands), scored=[s for b in bands for s in b["scored"]],
                ttfd=[s for b in bands for s in b["ttfd"]], fa=sum(b["fa"] for b in bands),
                minutes=sum(b["minutes"] for b in bands))


def band_row(r):
    """The reported metrics for one band, from its raw counts."""
    scored = sorted(r["scored"], key=lambda s: -s[0])
    conf = np.array([c for c, _ in scored])
    tp = np.array([t for _, t in scored], dtype=bool)
    keep = conf >= CONF  # operating point
    n_tp, n_fp = tp[keep].sum(), (~tp[keep]).sum()
    hits = [s for s in r["ttfd"] if s is not None]
    ap = float("nan")
    if r["n_gt"] and len(tp):
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        ap = compute_ap(ctp / r["n_gt"], ctp / (ctp + cfp))[0]
    return {
        "vehicles": len(r["ttfd"]),
        "GT boxes": r["n_gt"],
        "detection rate": n_tp / r["n_gt"] if r["n_gt"] else float("nan"),
        "precision": n_tp / (n_tp + n_fp) if n_tp + n_fp else float("nan"),
        "false alarms/min": r["fa"] / r["minutes"],
        "vehicles detected": f"{len(hits)}/{len(r['ttfd'])}",
        "TTFD median, s": float(np.median(hits)) if hits else float("nan"),
        "TTFD max, s": max(hits) if hits else float("nan"),
        "AP50": ap,
    }


def to_markdown(table):
    """GitHub table: clip name on its first row only, NaN as '–' (no vehicles in that band)."""
    fmt = {"detection rate": "{:.3f}", "precision": "{:.3f}", "false alarms/min": "{:.1f}",
           "TTFD median, s": "{:.2f}", "TTFD max, s": "{:.2f}", "AP50": "{:.3f}"}
    lines = ["| " + " | ".join(["clip", "band", *table.columns]) + " |",
             "|" + "|".join(["---"] * 2 + ["---:"] * len(table.columns)) + "|"]
    prev = None
    for (clip, band), row in table.iterrows():
        cells = [f"**{clip}**" if clip != prev else "", band]
        cells += ["–" if isinstance(v, float) and math.isnan(v) else fmt.get(c, "{}").format(v) for c, v in row.items()]
        lines.append("| " + " | ".join(cells) + " |")
        prev = clip
    return "\n".join(lines) + "\n"


raw = {name: score_clip(name, cfg) for name, cfg in CLIPS.items()}
raw["all clips"] = [merge([raw[c][i] for c in CLIPS]) for i in range(len(BANDS))]
print(f"\noperating point: conf >= {CONF:.3f} (best val F1), match IoU >= {IOU_THR}")
metrics_table = pd.DataFrame([
    {"clip": clip, "band": band, **band_row(r)} for clip, rs in raw.items() for (band, _, _), r in zip(BANDS, rs)
]).set_index(["clip", "band"])
md_path = ROOT / "metrics-table.md"
md_path.write_text(to_markdown(metrics_table) + f"\nOperating point: conf ≥ {CONF:.3f} (best F1 on val), a prediction "
                   f"matches a vehicle at IoU ≥ {IOU_THR}. – = no vehicles in that band.\n", encoding="utf-8")
print(f"markdown table -> {md_path}")
metrics_table.round(3)

[close-up] 1080x1080 crop of 1920x1080, 30.00 fps, 30.6 s, f = 1223 px, longer-side thresholds: 55.0 px @ 100 m, 13.8 px @ 400 m
    vehicles per band: 10 0-100 m, 0 100-400 m, 0 beyond 400 m
    unmatched predictions @ CONF: 1333 duplicates of a found vehicle, 726 badly placed on a vehicle, 530 on background -> 82 false-alarm events
[wide-shot] 1080x1080 crop of 1920x1080, 30.00 fps, 8.1 s, f = 1223 px, longer-side thresholds: 55.0 px @ 100 m, 13.8 px @ 400 m
    vehicles per band: 6 0-100 m, 11 100-400 m, 0 beyond 400 m
    unmatched predictions @ CONF: 8 duplicates of a found vehicle, 551 badly placed on a vehicle, 180 on background -> 107 false-alarm events

operating point: conf >= 0.429 (best val F1), match IoU >= 0.5


vehicles  GT boxes  detection rate  precision  \
clip      band                                                       
close-up  0-100 m          10      1546           0.878      0.358   
          100-400 m         0         0             NaN      0.000   
wide-shot 0-100 m           6       549           0.115      0.364   
          100-400 m        11      1150           0.264      0.324   
all clips 0-100 m          16      2095           0.678      0.358   
          100-400 m        11      1150           0.264      0.259   

                     false alarms/min vehicles detected  TTFD median, s  \
clip      band                                                            
close-up  0-100 m              68.627             10/10           0.000   
          100-400 m            92.157               0/0             NaN   
wide-shot 0-100 m               7.438               4/6           0.283   
          100-400 m           773.554              9/11           0.033   
all clips 0-100 m              55.862             14/16           0.000   
          100-400 m           234.310              9/11           0.033   

                     TTFD max, s   AP50  
clip      band                           
close-up  0-100 m          0.700  0.729  
          100-400 m          NaN    NaN  
wide-shot 0-100 m          4.133  0.156  
          100-400 m        0.800  0.198  
all clips 0-100 m          4.133  0.579  
          100-400 m        0.800  0.146